In [5]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from utils.bam import SymbolDataset, SymbolClassifier

In [ ]:
# Classifier Parameters
sf = 9
input = 256
hidden = 1024
output = 2 ** sf
lr=1e-3
batch_size=32

folder_path = "classifier_dataset_sf{}_{}_{}".format(sf, input, output)

X = np.load(f"{folder_path}/X.npy")   # shape (30720, 16)
y = np.load(f"{folder_path}/y.npy")   # shape (30720,)

dataset = SymbolDataset(X, y)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

layers = [input,hidden,output]
model = SymbolClassifier(layers).to(device)
criterion = nn.CrossEntropyLoss()   # Softmax included
optimizer = optim.AdamW(model.parameters(), lr=lr,weight_decay=1e-4) #weight_decay=1e-4 optim.Adam

In [ ]:
num_epochs = 75

for epoch in range(num_epochs):
    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in dataloader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_batch)              # (batch, 512)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * X_batch.size(0)

        preds = torch.argmax(logits, dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

    avg_loss = total_loss / total
    acc = correct / total * 100

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Loss: {avg_loss:.4f} | Accuracy: {acc:.2f}%")
    
torch.save(model.state_dict(), "symbol_classifier.pt")

Epoch [1/75] Loss: 3.2023 | Accuracy: 13.35%
Epoch [2/75] Loss: 2.5684 | Accuracy: 35.90%
Epoch [3/75] Loss: 2.2785 | Accuracy: 41.29%
Epoch [4/75] Loss: 2.1307 | Accuracy: 43.31%
Epoch [5/75] Loss: 2.0248 | Accuracy: 45.92%
Epoch [6/75] Loss: 1.9000 | Accuracy: 46.40%
Epoch [7/75] Loss: 1.8207 | Accuracy: 47.43%
Epoch [8/75] Loss: 1.7597 | Accuracy: 48.96%
Epoch [9/75] Loss: 1.6988 | Accuracy: 49.36%
Epoch [10/75] Loss: 1.6526 | Accuracy: 49.93%
Epoch [11/75] Loss: 1.6032 | Accuracy: 50.50%
Epoch [12/75] Loss: 1.5605 | Accuracy: 51.57%
Epoch [13/75] Loss: 1.5225 | Accuracy: 52.40%
Epoch [14/75] Loss: 1.5003 | Accuracy: 52.64%
Epoch [15/75] Loss: 1.4483 | Accuracy: 53.75%
Epoch [16/75] Loss: 1.4116 | Accuracy: 54.65%
Epoch [17/75] Loss: 1.4052 | Accuracy: 55.13%
Epoch [18/75] Loss: 1.3644 | Accuracy: 55.50%
Epoch [19/75] Loss: 1.3378 | Accuracy: 56.23%
Epoch [20/75] Loss: 1.3155 | Accuracy: 57.20%
Epoch [21/75] Loss: 1.2994 | Accuracy: 57.67%
Epoch [22/75] Loss: 1.2609 | Accuracy: 58.3

In [ ]:

def evaluate(model, dataloader):
    model.eval()
    correct1 = 0
    correct2 = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in dataloader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)

            _, top2 = torch.topk(logits, k=2, dim=1)
            preds = torch.argmax(logits, dim=1)

            correct1 += (preds == y_batch).sum().item()
            correct2 += (top2 == y_batch.unsqueeze(1)).any(dim=1).sum().item()
            total += y_batch.size(0)

    print(f"Top-1 Accuracy: {100*correct1/total:.2f}%")
    print(f"Top-2 Accuracy: {100*correct2/total:.2f}%")


# Load

model.load_state_dict(torch.load("symbol_classifier.pt"))
model.eval()
evaluate(model,dataloader)

Top-1 Accuracy: 43.72%
Top-2 Accuracy: 69.78%
